# Set up

In [1]:
import sys
if sys.platform == 'linux':
    sys.path.append("/home/qix/MultiNeuronGLM")
else:
    sys.path.append("D:/Github/MultiNeuronGLM")

In [57]:
import pandas as pd
import utility_functions as utils
import GLM
from DataLoader import Allen_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
sns.set_theme()

In [62]:
# Load LFP data
start_time = 0.0
end_time = 0.50
padding = 0.1
V1 = Allen_dataset(fps=1000,
               start_time=start_time, 
               end_time=end_time,
               padding=padding,
#                    orientation=[0],
               session_id=757216464,
               selected_probes=['probeA', 'probeB', 'probeC', 'probeD', 'probeE', 'probeF'],
#                    temporal_frequency=[1,2,4],
               stimulus_condition_id=[275, 277, 246, 255, 272, 248, 283, 266, 274, 276, 286, 271, 268, 270],
#                stimulus_condition_id = [246, 247, 248, 249, 250, 251, 252, 253, 255, 256, 257, 258,
#                                        259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 271,
#                                        272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284,
#                                        285, 286, 270],
               stimulus_name='drifting_gratings')

# V1.get_lfp()
# V1.remove_padding(padding)
V1.get_trial_metric_per_unit_per_trial()
V1.get_trial_metric_per_unit_per_trial(metric_type='spike_times')
V1.get_running(method="mine")

/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/stimulus_table/naming_utilities.py:154: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  movie_rows = table[stim_colname].str.contains(movie_re, na=False)
/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/ecephys_session.py:1315: UserWarning: Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',Spikes within these intervals are invalid and may need to be excluded from the analysis.
  warnings.warn("Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',"


In [63]:
# Load selected group_id
import pickle
with open('group_id_selected_a_c/membership.pickle', 'rb') as handle:
    membership = pickle.load(handle)
with open('group_id_selected_a_c/condition_ids.pickle', 'rb') as handle:
    condition_ids = pickle.load(handle)

In [385]:
select_trials = V1.running_trial_index
num_basis_baseline = 10

model = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model.add_effect('inhomogeneous_baseline', num=num_basis_baseline, add_constant_basis=False)
model.add_effect('coupling', 'probeD', peaks_max=150, num=5, nonlinear=0.2 )
model.add_effect('coupling', 'probeE', peaks_max=150, num=5, nonlinear=0.2 )
model.fit('probeC', verbose=True)

Negative log likelihood is: 28071.10
aic/2 is: 28091.10


In [106]:
import statsmodels.api as sm

# Make sure statsmodels and my code gives the same results

## statsmodels

In [421]:
beta = sm.GLM(model.response, model.predictors, family=sm.families.Poisson()).fit().params

In [422]:
beta

array([-2.14834214e+00, -8.50190472e-02, -1.47238708e+00, -1.56260303e+00,
       -1.16756118e+00, -1.73309236e+00, -1.73224732e+00, -1.57211894e+00,
       -1.57507020e+00, -2.32754108e+00,  2.74496749e-02,  1.87430258e-02,
        1.70411002e-02, -2.33954263e-02,  2.43029193e-02,  7.76782496e-02,
       -2.84670132e-02,  1.40034770e-03,  2.57664605e-03, -1.37590198e-03])

In [425]:
sm.GLM(model.response, model.predictors, family=sm.families.Poisson()).fit().bse

array([0.07624667, 0.0355798 , 0.04987483, 0.05073488, 0.0455692 ,
       0.04351236, 0.04337238, 0.04325843, 0.04677698, 0.0753995 ,
       0.01031109, 0.01055765, 0.00826904, 0.00565529, 0.00263409,
       0.00838834, 0.00827443, 0.00632973, 0.00420389, 0.00182255])

In [423]:
model.log_lmbd.flatten('F')

array([-2.14834214, -2.08430225, -2.021773  , ..., -1.59992223,
       -1.61521368, -1.59345898])

In [424]:
model.nll + + L2_pen * np.linalg.norm(beta*penalty_vec)**2

28071.09704040041

## 

In [435]:
L2_pen = 0
penalty_vec = np.ones((model.predictors.shape[1], 1))
penalty_vec[0] = 0
result = GLM.poisson_regression(model.response, model.predictors, L2_pen = L2_pen)

In [436]:
result.param

array([-2.14834208e+00, -8.50190592e-02, -1.47238706e+00, -1.56260302e+00,
       -1.16756117e+00, -1.73309235e+00, -1.73224731e+00, -1.57211893e+00,
       -1.57507020e+00, -2.32754106e+00,  2.74496755e-02,  1.87430247e-02,
        1.70411004e-02, -2.33954261e-02,  2.43029190e-02,  7.76782499e-02,
       -2.84670136e-02,  1.40034782e-03,  2.57664614e-03, -1.37590196e-03])

In [437]:
result.bse

array([0.07624667, 0.0355798 , 0.04987483, 0.05073488, 0.0455692 ,
       0.04351236, 0.04337238, 0.04325843, 0.04677698, 0.0753995 ,
       0.01031109, 0.01055765, 0.00826904, 0.00565529, 0.00263409,
       0.00838834, 0.00827443, 0.00632973, 0.00420389, 0.00182255])

In [418]:
log_lmbda_hat.squeeze()

array([-2.14834208, -2.08430219, -2.02177294, ..., -1.59992221,
       -1.61521367, -1.59345897])

In [419]:
GLM.spike_trains_neg_log_likelihood(log_lmbda_hat, model.response[:,np.newaxis]) \
    + L2_pen * np.linalg.norm(beta*penalty_vec)**2

28071.09704040041

In [420]:
L2_pen * np.linalg.norm(beta*penalty_vec)**2

0.0

# old version

In [23]:
sys.path.append("/home/qix/MultiNeuronGLM/IPRFfunctions/")
import smoothing_spline

In [41]:
spikes = V1.spike_train.iloc[15,0][np.newaxis, :]
spikes = np.vstack((spikes, spikes))
time_line = V1.time_line
spmodel = smoothing_spline.SmoothingSpline()
log_lambda_hat, (beta, beta_baseline, log_lambda_offset, hessian, hessian_baseline, nll) \
    = spmodel.poisson_regression_smoothing_spline(spikes, time_line, num_knots=10)


In [32]:
basis, Omega = spmodel.construct_basis_omega(time_line, knots=10)

In [37]:
GLM.spike_trains_neg_log_likelihood(np.array([1,0,1])[:,np.newaxis],
                                   np.array([0.5,0.3,0.5])[:,np.newaxis], 
                                   trial_wise=True)

array([5.43656366])

In [134]:
aaa = np.array([1,0,1])
bbb = np.array([0.5,0.3,0.5])[:,np.newaxis]
GLM.spike_trains_neg_log_likelihood(aaa,
                                   np.hstack((bbb,bbb)), 
                                   trial_wise=True)

10.87312731383618

True

In [130]:
ccc = np.zeros((10, 1))
ccc[0] = 1
ccc[1:]

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])